# Week 8 Capstone — Meeting-to-Action Assistant

This is the **self-contained submission notebook** for COSC 650. It takes an existing meeting transcript and produces a structured operational brief containing an executive summary, decisions, action items and owners, blockers, and follow-up questions.

## Rubric coverage

- **Integration by reuse:** Week 3 structured prompting, Week 4 deterministic validation, Week 5 MiniLM + FAISS retrieval, Week 6 cross-encoder reranking, and Week 7 dynamic few-shot selection.
- **Evaluation:** 10 held-out cases, concrete extraction metrics, structured-output validity, latency, and an LLM-as-judge pass on correctness, grounding, and operational usefulness.
- **Required failure mode:** an A/B/C ablation traces a suggestion-as-decision error to retrieval, few-shot context, or generation; a deterministic mitigation is measured.
- **Architecture and rationale:** documented here and in `week-08/README.md`.
- **Portfolio curation:** this notebook and `week-08/research-note.md` are linked from the repository README.

```text
transcript → MiniLM/FAISS retrieval → cross-encoder rerank → optional few-shot examples → Gemini Flash → deterministic validation → evaluation
```

The notebook embeds its dataset and implementation directly. No project-local Python module or external dataset is required. The scope is transcript text in and an evaluated operational brief out; transcription, Teams/Zoom integration, automatic email/task execution, fine-tuning, deployment, web UI, and multi-agent orchestration are out of scope.

In [ ]:
# In Colab, uncomment once if needed:
# %pip install -q sentence-transformers faiss-cpu openai pandas numpy

import os, re, json, time, math, hashlib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss

try:
    from google.colab import userdata
    if not os.getenv('GEMINI_API_KEY'):
        key = userdata.get('GEMINI_API_KEY')
        if key: os.environ['GEMINI_API_KEY'] = key
except Exception:
    pass

GEMINI_MODEL = os.getenv('GEMINI_MODEL', 'gemini-3.5-flash')
GEMINI_JUDGE_MODEL = os.getenv('GEMINI_JUDGE_MODEL', GEMINI_MODEL)
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
RERANK_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
MAX_SYNTHESIS_CALLS = 30
MAX_JUDGE_CALLS = 30

# Optional: after a measured run, paste the printed CACHE_EXPORT string here before a rerun.
# This keeps the notebook itself self-contained while avoiding repeated free-tier calls.
PRELOADED_CACHE_JSON = ''
CACHE = json.loads(PRELOADED_CACHE_JSON) if PRELOADED_CACHE_JSON.strip() else {'synthesis': {}, 'judge': {}}
LIVE_SYNTHESIS = 0
LIVE_JUDGE = 0

# Ten held-out cases. These are not used as few-shot demonstrations.
EVAL_CASES = [
 {'id':'E01','t':"Brittany: We need the renewal demo ready by Friday.\nMichael: I can finish the validation worksheet Thursday.\nBrittany: Good. Michael owns the worksheet, and we'll use the current pricing template for the demo.\nMichael: The OpenAir export is still missing two customer IDs.\nBrittany: Ask Finance whether those IDs can be corrected before Thursday.",'d':['Use the current pricing template for the renewal demo.'],'a':[('Finish the validation worksheet by Thursday.','Michael'),('Ask Finance whether the missing customer IDs can be corrected before Thursday.','Michael')],'b':['The OpenAir export is missing two customer IDs.'],'q':['Can Finance correct the missing customer IDs before Thursday?']},
 {'id':'E02','t':"Alex: Maybe we should move the launch to Monday.\nPriya: I would rather keep Friday unless QA finds something serious.\nAlex: Agreed. No schedule change right now.\nPriya: I'll send the final QA result by 3 PM.",'d':['Keep the Friday launch schedule unless QA finds a serious issue.'],'a':[('Send the final QA result by 3 PM.','Priya')],'b':[],'q':[]},
 {'id':'E03','t':"Sam: The customer asked whether SSO can be included in phase one.\nJordan: We don't know the engineering effort yet.\nSam: Could we promise it conditionally?\nJordan: No. Let's get an estimate first. Jordan will ask Engineering for the effort by Wednesday.",'d':['Do not promise SSO for phase one until an engineering estimate is available.'],'a':[('Ask Engineering for the SSO effort estimate by Wednesday.','Jordan')],'b':['The engineering effort for SSO is unknown.'],'q':['What is the engineering effort for adding SSO in phase one?']},
 {'id':'E04','t':"Nina: I think a weekly dashboard could help.\nOmar: Possibly. Let's look at examples next week.\nNina: There is no decision on dashboard cadence today.\nOmar: I'll collect three dashboard examples before our next meeting.",'d':[],'a':[('Collect three dashboard examples before the next meeting.','Omar')],'b':[],'q':[]},
 {'id':'E05','t':"Lee: The import failed because three rows have invalid dates.\nCasey: Can we fix them manually?\nLee: Yes. I'll correct the dates and rerun the import today.\nCasey: Approved. Keep the current import mapping; don't redesign it for this release.",'d':['Keep the current import mapping for this release.'],'a':[('Correct the invalid dates and rerun the import today.','Lee')],'b':['Three rows have invalid dates and caused the import to fail.'],'q':[]},
 {'id':'E06','t':"Maya: We have two vendors that could supply the API data.\nChris: Vendor B is cheaper, but legal hasn't reviewed its terms.\nMaya: Let's not choose a vendor yet.\nChris: I'll send both agreements to Legal.\nMaya: Which agreement can Legal clear this week?",'d':['Do not choose an API data vendor yet.'],'a':[('Send both vendor agreements to Legal.','Chris')],'b':["Legal has not reviewed Vendor B's terms."],'q':['Which vendor agreement can Legal clear this week?']},
 {'id':'E07','t':"Dana: The pilot users liked the shorter form.\nEli: Should we remove the optional notes field too?\nDana: Not yet. We did agree to keep the shorter form for the pilot.\nEli: I'll measure completion time for another week.",'d':['Keep the shorter form for the pilot.'],'a':[('Measure form completion time for another week.','Eli')],'b':[],'q':['Should the optional notes field be removed?']},
 {'id':'E08','t':"Rosa: The report numbers from Analytics and Search Console don't match.\nIan: The date ranges are different.\nRosa: Use September 1 through September 30 for both sources.\nIan: Got it. I'll rerun both reports with that date range and send the numbers tomorrow.",'d':['Use September 1 through September 30 as the date range for both sources.'],'a':[('Rerun both reports with the September 1 through September 30 date range and send the numbers tomorrow.','Ian')],'b':['Analytics and Search Console report numbers do not match because the date ranges differ.'],'q':[]},
 {'id':'E09','t':"Taylor: We could automate the email after generation.\nMorgan: I don't want automatic sending in version one.\nTaylor: Okay, generated drafts will require human review.\nMorgan: Taylor, document that approval gate in the design.",'d':['Version one will not automatically send generated email; generated drafts require human review.'],'a':[('Document the human approval gate in the design.','Taylor')],'b':[],'q':[]},
 {'id':'E10','t':"Avery: The customer list has duplicate organizations.\nQuinn: That explains the mismatched renewal totals.\nAvery: We need a matching key before proposal generation.\nQuinn: I'll test domain plus customer ID as the matching key.\nAvery: Is customer ID populated for every record?",'d':['A matching key is required before proposal generation.'],'a':[('Test domain plus customer ID as the matching key.','Quinn')],'b':['Duplicate organizations are causing mismatched renewal totals.'],'q':['Is customer ID populated for every record?']}
]
assert len(EVAL_CASES) == 10

RULES = [
 'A decision requires commitment, approval, agreement, selection, rejection, or an explicit commitment.',
 'A suggestion, possibility, brainstorm, maybe/could/might statement is not a decision unless a later turn commits to it.',
 'An action item is future work assigned or accepted by a person or team; do not invent an owner.',
 "Owner resolution may require cross-speaker context; 'I will' or 'I'll' refers to the current speaker.",
 'A blocker is a concrete condition preventing, delaying, or risking progress.',
 'A follow-up question must remain unresolved at the end of the transcript.',
 'Every operational item must be grounded in transcript evidence.'
]
FEW_SHOT = [
 ('Maybe delay launch. No, keep Friday.','Keep Friday; the delay was only a suggestion.'),
 ('Could we use vendor B? Let us compare legal terms first.','No vendor decision was made.'),
 ('The export is missing IDs. I will ask Finance to fix them.','Action item: ask Finance to fix the IDs.'),
 ('Legal has not reviewed the vendor terms.','Blocker: legal review is outstanding.'),
 ('Which agreement can Legal clear this week?','Unresolved follow-up question.'),
 ('We agreed to require human review before sending.','Decision: human review is required before sending.'),
 ('Maybe remove the notes field. Not yet.','The removal was discussed but not approved.'),
 ('Use September 1 through September 30 for both reports.','Decision: both reports use that date range.')
]

embedder = SentenceTransformer(EMBED_MODEL)
reranker = CrossEncoder(RERANK_MODEL)
rule_vecs = embedder.encode(RULES, normalize_embeddings=True).astype('float32')
rule_index = faiss.IndexFlatIP(rule_vecs.shape[1]); rule_index.add(rule_vecs)
few_vecs = embedder.encode([x[0] for x in FEW_SHOT], normalize_embeddings=True).astype('float32')
few_index = faiss.IndexFlatIP(few_vecs.shape[1]); few_index.add(few_vecs)

def retrieve(query, docs, index, text_fn=lambda x:x, top_k=4):
    q = embedder.encode([query], normalize_embeddings=True).astype('float32')
    _, ids = index.search(q, min(6, len(docs)))
    cand = [docs[int(i)] for i in ids[0] if i >= 0]
    scores = reranker.predict([(query, text_fn(x)) for x in cand])
    return [x for _,x in sorted(zip(scores,cand), key=lambda z:z[0], reverse=True)[:top_k]]

def key_for(x): return hashlib.sha256(json.dumps(x, sort_keys=True).encode()).hexdigest()
def parse_json(s):
    s = re.sub(r'^```(?:json)?\s*','',s.strip(),flags=re.I); s = re.sub(r'\s*```$','',s)
    return json.loads(s)

def api_client():
    from openai import OpenAI
    if not os.getenv('GEMINI_API_KEY'): raise RuntimeError('GEMINI_API_KEY is not set')
    return OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')

def validate(x):
    req={'executive_summary':str,'decisions':list,'action_items':list,'blockers':list,'follow_up_questions':list}
    return isinstance(x,dict) and all(k in x and isinstance(x[k],t) for k,t in req.items())

PROMPT = '''Return JSON only. Convert the transcript into an operational brief with this exact shape:
{"executive_summary":"string","decisions":[{"decision":"string","evidence":"string"}],"action_items":[{"description":"string","owner":"string","evidence":"string"}],"blockers":[{"blocker":"string","evidence":"string"}],"follow_up_questions":[{"question":"string","evidence":"string"}]}
Report only confirmed decisions, never suggestions. Do not invent owners. A blocker must materially delay/prevent/risk progress. Follow-up questions must still be unresolved. Include transcript evidence for every item.'''

def extract(transcript, config):
    global LIVE_SYNTHESIS
    ck=key_for({'m':GEMINI_MODEL,'c':config,'t':transcript})
    if ck in CACHE['synthesis']:
        out=dict(CACHE['synthesis'][ck]); out['cache_hit']=True; return out
    if LIVE_SYNTHESIS >= MAX_SYNTHESIS_CALLS: raise RuntimeError('Synthesis call cap reached')
    parts=[PROMPT]; log={'rules':[],'examples':[]}
    if config in ('B','C'):
        log['rules']=retrieve(transcript,RULES,rule_index); parts.append('Retrieved guidance:\n'+'\n'.join('- '+x for x in log['rules']))
    if config=='C':
        log['examples']=retrieve(transcript,FEW_SHOT,few_index,lambda x:x[0],3); parts.append('Examples:\n'+'\n'.join('- '+a+' => '+b for a,b in log['examples']))
    parts.append('Transcript:\n'+transcript); t0=time.perf_counter()
    r=api_client().chat.completions.create(model=GEMINI_MODEL,messages=[{'role':'user','content':'\n\n'.join(parts)}],temperature=0)
    raw=r.choices[0].message.content; latency=time.perf_counter()-t0
    try: brief=parse_json(raw)
    except Exception: brief={}
    out={'brief':brief,'valid':validate(brief),'latency_s':latency,'retrieval':log}
    CACHE['synthesis'][ck]=out; LIVE_SYNTHESIS += 1; out=dict(out); out['cache_hit']=False; return out

STOP={'the','a','an','to','of','and','or','for','in','on','by','with','is','are','be','as','at','this','that'}
def toks(s): return {x for x in re.findall(r'[a-z0-9]+',str(s).lower()) if x not in STOP}
def jac(a,b):
    A,B=toks(a),toks(b); return len(A&B)/len(A|B) if A and B else float(not A and not B)
def prf(pred,gold,thr=.42):
    unused=set(range(len(gold))); tp=0; pairs=[]
    for pi,p in enumerate(pred):
        scores=[(jac(p,gold[i]),i) for i in unused]
        if scores:
            s,gi=max(scores)
            if s>=thr: tp+=1; pairs.append((pi,gi)); unused.remove(gi)
    P=tp/len(pred) if pred else (1.0 if not gold else 0.0); R=tp/len(gold) if gold else (1.0 if not pred else 0.0)
    return P,R,(2*P*R/(P+R) if P+R else 0.0),pairs
def score(case,res):
    b=res['brief'] if res.get('valid') else {}; d=[x.get('decision','') for x in b.get('decisions',[])]; a=[x.get('description','') for x in b.get('action_items',[])]; owners=[x.get('owner','') for x in b.get('action_items',[])]; bl=[x.get('blocker','') for x in b.get('blockers',[])]; q=[x.get('question','') for x in b.get('follow_up_questions',[])]
    dp,dr,df,_=prf(d,case['d']); ap,ar,af,pairs=prf(a,[x[0] for x in case['a']]); _,_,bf,_=prf(bl,case['b']); _,_,qf,_=prf(q,case['q'])
    hits=[owners[pi].lower()==case['a'][gi][1].lower() for pi,gi in pairs if pi<len(owners)]
    owner=sum(hits)/len(hits) if hits else 0.0
    return {'case_id':case['id'],'decision_precision':dp,'decision_recall':dr,'decision_f1':df,'action_precision':ap,'action_recall':ar,'action_f1':af,'owner_accuracy':owner,'blocker_f1':bf,'question_f1':qf,'valid':float(res.get('valid',False)),'latency_s':res.get('latency_s',math.nan)}

print('Held-out cases:',len(EVAL_CASES)); print('Gemini:',GEMINI_MODEL); print('Judge:',GEMINI_JUDGE_MODEL)

## A/B/C ablation

- **A:** transcript only
- **B:** transcript + retrieved/reranked operational guidance
- **C:** transcript + retrieved/reranked guidance + dynamic few-shot examples

In [ ]:
ABLATION_RESULTS={}; rows=[]
if not os.getenv('GEMINI_API_KEY'):
    print('API-BLOCKED: add GEMINI_API_KEY and rerun.')
else:
    for cfg in ['A','B','C']:
        ABLATION_RESULTS[cfg]={}
        for case in EVAL_CASES:
            r=extract(case['t'],cfg); ABLATION_RESULTS[cfg][case['id']]=r
            s=score(case,r); s['config']=cfg; s['cache_hit']=r['cache_hit']; rows.append(s)
    METRICS=pd.DataFrame(rows); display(METRICS.round(3)); display(METRICS.groupby('config').mean(numeric_only=True).round(3))

## LLM-as-judge

All 10 held-out cases are judged for each configuration on exactly three named dimensions: **correctness**, **grounding**, and **operational usefulness**.

In [ ]:
JUDGE_DF=pd.DataFrame()
JUDGE_PROMPT='Score the candidate 1-5 on correctness, grounding, and operational_usefulness. Return JSON only: {"correctness":1,"grounding":1,"operational_usefulness":1,"reason":"brief reason"}'
def judge(case,candidate):
    global LIVE_JUDGE
    payload={'transcript':case['t'],'gold':{'decisions':case['d'],'actions':case['a'],'blockers':case['b'],'questions':case['q']},'candidate':candidate}; ck=key_for({'m':GEMINI_JUDGE_MODEL,'p':payload})
    if ck in CACHE['judge']: out=dict(CACHE['judge'][ck]); out['cache_hit']=True; return out
    if LIVE_JUDGE>=MAX_JUDGE_CALLS: raise RuntimeError('Judge call cap reached')
    r=api_client().chat.completions.create(model=GEMINI_JUDGE_MODEL,messages=[{'role':'user','content':JUDGE_PROMPT+'\n\n'+json.dumps(payload)}],temperature=0)
    out=parse_json(r.choices[0].message.content); CACHE['judge'][ck]=out; LIVE_JUDGE+=1; out=dict(out); out['cache_hit']=False; return out
if ABLATION_RESULTS:
    jr=[]
    for cfg in ['A','B','C']:
        for case in EVAL_CASES: jr.append({'config':cfg,'case_id':case['id'],**judge(case,ABLATION_RESULTS[cfg][case['id']]['brief'])})
    JUDGE_DF=pd.DataFrame(jr); display(JUDGE_DF); display(JUDGE_DF.groupby('config')[['correctness','grounding','operational_usefulness']].mean().round(3))

## Required failure mode and mitigation

The declared failure is a suggestion or discussion point being incorrectly promoted to a confirmed decision. The trace first looks for a B→C decision-F1 regression (few-shot context), then A→B (retrieval context). If neither occurs, the lowest-scoring case is attributed to generation rather than inventing a retrieval failure.

In [ ]:
FAILURE=None
if ABLATION_RESULTS:
    scores={cfg:{c['id']:score(c,ABLATION_RESULTS[cfg][c['id']]) for c in EVAL_CASES} for cfg in ['A','B','C']}
    chosen=component=None
    for c in EVAL_CASES:
        if scores['C'][c['id']]['decision_f1'] < scores['B'][c['id']]['decision_f1']: chosen,component=c['id'],'dynamic few-shot context'; break
    if chosen is None:
        for c in EVAL_CASES:
            if scores['B'][c['id']]['decision_f1'] < scores['A'][c['id']]['decision_f1']: chosen,component=c['id'],'retrieved/reranked context'; break
    if chosen is None: chosen=min((c['id'] for c in EVAL_CASES),key=lambda x:scores['C'][x]['decision_f1']); component='generation/extraction stage'
    FAILURE=(chosen,component); print('Failure case:',chosen); print('Responsible component:',component)
    for cfg in ['A','B','C']: print('\n',cfg,json.dumps(ABLATION_RESULTS[cfg][chosen]['brief'],indent=2),'\nscore:',scores[cfg][chosen])

HEDGE=['maybe','could','might','consider','possibly','what if','i think','would rather']; COMMIT=['agreed','approved','decided','we will',"we'll",'keep','use ','do not',"don't",'required','requires']
def mitigate(brief):
    out=json.loads(json.dumps(brief)); out['decisions']=[d for d in out.get('decisions',[]) if not (any(h in d.get('evidence','').lower() for h in HEDGE) and not any(c in d.get('evidence','').lower() for c in COMMIT))]; return out
if ABLATION_RESULTS:
    before=[]; after=[]
    for case in EVAL_CASES:
        r=ABLATION_RESULTS['C'][case['id']]; before.append(score(case,r)); rr=dict(r); rr['brief']=mitigate(r['brief']); rr['valid']=validate(rr['brief']); after.append(score(case,rr))
    print('Decision F1 before mitigation:',round(pd.DataFrame(before).decision_f1.mean(),3)); print('Decision F1 after mitigation:',round(pd.DataFrame(after).decision_f1.mean(),3))

## Submission guard

After the measured run, save this notebook with outputs, update `week-08/research-note.md` with the actual measurements and final architecture decision, and export this executed notebook to HTML. The next cell also prints the cache as a JSON string; paste it into `PRELOADED_CACHE_JSON` before committing if you want future clean reruns to avoid repeating Gemini calls while keeping the notebook self-contained.

In [ ]:
problems=[]
if not ABLATION_RESULTS: problems.append('A/B/C evaluation missing')
if JUDGE_DF.empty or len(JUDGE_DF)!=len(EVAL_CASES)*3: problems.append('judge pass missing/incomplete')
if FAILURE is None: problems.append('failure trace missing')
if problems:
    print('NOT READY FOR SUBMISSION'); [print('-',p) for p in problems]
else:
    print('READY FOR SUBMISSION'); print('- 10 held-out cases × 3 configurations'); print('- deterministic metrics complete'); print('- LLM-as-judge complete'); print('- failure traced and mitigation measured')
print('\nCACHE_EXPORT =', json.dumps(CACHE, separators=(',',':')))
print('\nHTML export: jupyter nbconvert --to html week8_meeting_to_action_capstone.ipynb')